# QSage Tutorial

This tutorial demonstrates how to use **Quantum Sage (QSage)**, a meta-learning tool that predicts which quantum or classical ML model will perform best on your dataset.

## What is QSage?

QSage is a **surrogate model** trained on extensive benchmarking data that:

- **Predicts model performance** without running expensive experiments
- **Recommends best models** based on dataset characteristics
- **Saves computational resources** by avoiding trial-and-error
- **Supports both quantum and classical** ML algorithms

## How QSage Works

1. **Analyzes your dataset**: Computes data complexity metrics
2. **Predicts performance**: Uses trained surrogate model to estimate how each ML model will perform
3. **Ranks models**: Provides recommendations ordered by predicted performance
4. **Explains predictions**: Shows which data characteristics influenced the recommendation

## Tutorial Overview

In this tutorial, you will:

1. Download pre-trained QSage model weights
2. Load benchmark results data
3. Train QSage on the benchmark data (optional)
4. Use QSage to predict model performance on new datasets
5. Visualize and interpret QSage predictions

## 1. Setup and Imports

First, let's import the necessary modules.

In [ ]:
import pandas as pd
import sys
import os
import re
import math
import pickle

# Import QSage
from apps.sage.sage import QuantumSage

print("✓ Imports successful")

## 2. Download Pre-trained Model

QSage requires pre-trained model weights that were trained on extensive benchmarking data.

### Download the Model

**Download link**: [https://ibm.biz/QSageModel](https://ibm.biz/QSageModel)

The download includes:
- `QMLBench_sage.pkl`: Pre-trained Random Forest surrogate model
- `Compiled_QMLBench_results.csv`: Benchmark results used for training

### File Structure

Place the downloaded files in a `results/` directory:
```
results/
├── QMLBench_sage.pkl
└── Compiled_QMLBench_results.csv
```

In [ ]:
# Set paths to downloaded files
dir_home = ''  # Set to your QBioCode directory if needed
dir_results = os.path.join(dir_home, 'results')

file_input = os.path.join(dir_results, 'Compiled_QMLBench_results.csv')
file_sage = os.path.join(dir_results, 'QMLBench_sage.pkl')

# Sage model type (random_forest or mlp)
sage_type = 'random_forest'

print(f"Data file: {file_input}")
print(f"Model file: {file_sage}")
print(f"Sage type: {sage_type}")

## 3. Load Benchmark Data

The benchmark data contains results from running multiple ML models on various datasets.

### Data Structure

The benchmark data includes:
- **Dataset characteristics**: 20+ complexity metrics (intrinsic dimension, Fisher ratio, etc.)
- **Model results**: Accuracy, F1-score, AUC, training time
- **Model configurations**: Embeddings, hyperparameters
- **Multiple models**: Classical (RF, SVM, LR, DT, NB, MLP) and Quantum (PQK, QSVC, VQC, QNN)

In [ ]:
# Define features and metrics
features = [
    'Feature_Samples_ratio', 'Intrinsic_Dimension', 'Condition number',
    'Fisher Discriminant Ratio', 'Total Correlations', 'Mutual information',
    '# Non-zero entries', '# Low variance features', 'Variation', 'std_var',
    'Coefficient of Variation %', 'std_co_of_v', 'Skewness', 'std_skew',
    'Kurtosis', 'std_kurt', 'Mean Log Kernel Density',
    'Isomap Reconstruction Error', 'Fractal dimension', 'Entropy'
]
features.sort()

metrics = ['accuracy', 'f1_score', 'time', 'auc']
metrics.sort()

# Load benchmark results
reload = False  # Set to True to reprocess data

if reload or not os.path.exists(file_input):
    # Process raw results (if you have the raw data directory)
    path_to_compiled_results = dir_results
    if os.path.isdir(path_to_compiled_results):
        result = [os.path.join(dp, f) for dp, dn, filenames in os.walk(path_to_compiled_results) 
                  for f in filenames if ('csv' in f) and f != 'RawDataEvaluation.csv']
        results_df = []
        for fl in result:
            if 'Compiled' not in fl:
                results_df.append(pd.read_csv(fl))
        results_df = pd.concat(results_df)
        
        # Clean and process data
        for f in features:
            results_df[f] = [round(x, 4) for x in results_df[f]]
        results_df['embeddings'] = results_df['embeddings'].fillna('none')
        results_df['model'] = results_df['model'].fillna('none')
        results_df['datatype'] = [re.sub('\.csv', '', re.sub('-.*', '', str(x))) 
                                   for x in results_df['Dataset']]
        results_df['model_embed_datatype'] = [
            '_'.join([str(row.model), str(row.embeddings), str(row.datatype)]) 
            for idx, row in results_df.iterrows()
        ]
        
        results_df = results_df.reset_index(drop=True)
        results_df[results_df == math.inf] = 0
        results_df = results_df.drop_duplicates()
        results_df.to_csv(file_input)
    else:
        results_df = pd.read_csv(path_to_compiled_results)
else:
    # Load pre-compiled results
    results_df = pd.read_csv(file_input)

print(f"Loaded {len(results_df)} benchmark results")
print(f"Datasets: {results_df['Dataset'].nunique()}")
print(f"Models: {results_df['model'].nunique()}")
print(f"\nFirst few rows:")
results_df.head()

## 4. Train QSage (Optional)

If you want to train your own QSage model, you can do so here. Otherwise, skip to the next section to load the pre-trained model.

### Training Process

QSage training involves:
1. **Feature selection**: Choose which data complexity metrics to use
2. **Target selection**: Choose which performance metric to predict (accuracy, F1, AUC)
3. **Model training**: Train a Random Forest or MLP surrogate model
4. **Validation**: Evaluate prediction accuracy on held-out data

In [ ]:
# Select a held-out dataset for testing
held_out_dataset = 'iris_binary_class_dataset'

# Split data into training and held-out
train_df = results_df[~results_df['Dataset'].str.contains(held_out_dataset)]
held_out_df = results_df[results_df['Dataset'].str.contains(held_out_dataset)]

print(f"Training data: {len(train_df)} results")
print(f"Held-out data: {len(held_out_df)} results")

# Initialize QSage
sage = QuantumSage(
    data=train_df,
    features=features,
    metrics=metrics,
    sage_type=sage_type  # 'random_forest' or 'mlp'
)

print(f"\n✓ QSage initialized with {sage_type} model")

In [ ]:
# Train QSage
print("Training QSage...")
sage.train()
print("✓ Training complete!")

# Save trained model
with open(file_sage, 'wb') as f:
    pickle.dump(sage, f)
print(f"✓ Model saved to {file_sage}")

## 5. Load Pre-trained QSage Model

Load the pre-trained QSage model to make predictions.

In [ ]:
# Load pre-trained QSage model
with open(file_sage, 'rb') as f:
    sage = pickle.load(f)

print("✓ Pre-trained QSage model loaded")
print(f"Model type: {sage.sage_type}")
print(f"Features used: {len(sage.features)}")
print(f"Metrics predicted: {sage.metrics}")

## 6. Make Predictions with QSage

Now let's use QSage to predict which models will perform best on a new dataset.

### Prediction Process

1. **Provide dataset characteristics**: Data complexity metrics
2. **QSage predicts**: Performance for each model/embedding combination
3. **Get recommendations**: Models ranked by predicted performance

In [ ]:
# Prepare held-out dataset features
_columns_data_features = [
    '# Features', '# Samples', 'Feature_Samples_ratio', 'Intrinsic_Dimension',
    'Condition number', 'Fisher Discriminant Ratio', 'Total Correlations',
    'Mutual information', '# Non-zero entries', '# Low variance features',
    'Variation', 'std_var', 'Coefficient of Variation %', 'std_co_of_v',
    'Skewness', 'std_skew', 'Kurtosis', 'std_kurt', 'Mean Log Kernel Density',
    'Isomap Reconstruction Error', 'Fractal dimension', 'Entropy'
]

# Get unique dataset characteristics
held_out_features = held_out_df[_columns_data_features].drop_duplicates()

print("Held-out dataset characteristics:")
print(held_out_features)

# Make predictions
predictions = sage.predict(held_out_features)

print(f"\n✓ Generated predictions for {len(predictions)} model configurations")

## 7. Analyze Predictions

Let's examine QSage's predictions and compare them to actual results.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Merge predictions with actual results
comparison_df = held_out_df.merge(
    predictions,
    on=['model', 'embeddings'],
    suffixes=('_actual', '_predicted')
)

# Display top predictions
print("Top 10 Predicted Models (by accuracy):")
top_predictions = predictions.nlargest(10, 'accuracy_predicted')[
    ['model', 'embeddings', 'accuracy_predicted', 'f1_score_predicted']
]
print(top_predictions)

# Compare predictions vs actual
print("\nPrediction vs Actual Performance:")
comparison_summary = comparison_df[[
    'model', 'embeddings', 
    'accuracy_actual', 'accuracy_predicted',
    'f1_score_actual', 'f1_score_predicted'
]].head(10)
print(comparison_summary)

## 8. Visualize Results

Create visualizations to understand QSage's predictions.

In [ ]:
# Set style
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Predicted vs Actual Accuracy
axes[0].scatter(
    comparison_df['accuracy_predicted'],
    comparison_df['accuracy_actual'],
    alpha=0.6
)
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted Accuracy')
axes[0].set_ylabel('Actual Accuracy')
axes[0].set_title('QSage Prediction Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Model Rankings
top_models = predictions.nlargest(15, 'accuracy_predicted')
axes[1].barh(
    range(len(top_models)),
    top_models['accuracy_predicted'],
    color='steelblue'
)
axes[1].set_yticks(range(len(top_models)))
axes[1].set_yticklabels([f"{row.model}_{row.embeddings}" 
                          for _, row in top_models.iterrows()])
axes[1].set_xlabel('Predicted Accuracy')
axes[1].set_title('Top 15 Recommended Models')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Calculate prediction error
mae = abs(comparison_df['accuracy_actual'] - comparison_df['accuracy_predicted']).mean()
print(f"\nMean Absolute Error: {mae:.4f}")

## 9. Feature Importance

Understand which dataset characteristics most influence QSage's predictions.

In [ ]:
# Get feature importance (for Random Forest sage)
if sage.sage_type == 'random_forest':
    feature_importance = pd.DataFrame({
        'feature': sage.features,
        'importance': sage.model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot top 15 features
    plt.figure(figsize=(10, 6))
    top_features = feature_importance.head(15)
    plt.barh(range(len(top_features)), top_features['importance'], color='coral')
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title('Top 15 Most Important Features for QSage Predictions')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))
else:
    print("Feature importance analysis only available for Random Forest sage")

## Summary

In this tutorial, you learned how to:

1. ✅ Download and set up pre-trained QSage models
2. ✅ Load benchmark data with model performance results
3. ✅ Train QSage on benchmark data (optional)
4. ✅ Load pre-trained QSage models
5. ✅ Make predictions for new datasets
6. ✅ Analyze and visualize QSage predictions
7. ✅ Understand feature importance

## Key Takeaways

- **QSage saves time**: Predicts model performance without running expensive experiments
- **Data-driven recommendations**: Uses dataset characteristics to recommend best models
- **Interpretable**: Shows which features influence predictions
- **Flexible**: Supports both quantum and classical ML models

## Next Steps

- **Use QSage on your data**: Compute data complexity metrics and get model recommendations
- **Combine with QProfiler**: Use QSage to narrow down models, then validate with QProfiler
- **Retrain QSage**: Add your own benchmark results to improve predictions
- **Explore feature engineering**: Experiment with different data complexity metrics

## See Also

- [QSage Documentation](../../apps/sage.rst) - Full API reference
- [QProfiler Tutorial](../QProfiler/example_qprofiler.ipynb) - Benchmark models
- [Data Generation Tutorial](../Artificial_data_generation/example_data_generation.ipynb) - Create test datasets